# Download Sentinel-2 and dynamic world composites from GEE

Antonio Rangel  
Juan Terven  
2023

In this notebook, we download **Dynamic World** composites and **Sentinel-2** data.  

**Dynamic World** is a land cover dataset developed by Google and the World Resources Institute, designed to provide near real-time, high-resolution land classification maps using deep learning models applied to Sentinel-2 imagery. This dataset features nine land cover classes, including water, trees, grass, crops, shrubland, built-up areas, bare ground, snow/ice, and flooded vegetation, offering valuable insights for environmental monitoring and land-use analysis.  

**Sentinel-2** is a satellite mission from the European Space Agency (ESA) that provides multispectral imagery at high spatial and temporal resolutions. It is widely used in applications such as agriculture, forestry, land cover classification, and disaster monitoring. Sentinel-2's rich spectral information, combined with Dynamic World's classification capabilities, enables advanced analysis for various Earth observation tasks.  

In this notebook, we focus on retrieving these datasets from GEE to facilitate further analysis and applications in remote sensing and environmental research.

In [2]:
import ee
from ee.collection import ee_date
import folium
import time
import geemap

## Login to GEE

In [3]:
ee.Authenticate()
ee.Initialize(project='project name')

## Defining the Time Range and Region of Interest (ROI)

In this section, we define the temporal range and the geospatial region of interest (ROI) for extracting Dynamic World and Sentinel-2 data.

The **startDate** and **endDate** variables specify the time range for data retrieval, covering the period from July 1, 2019, to September 30, 2019.

The **geometry** variable defines a polygonal ROI using Earth Engine’s Geometry API. This polygon represents an area within the United States, encompassing multiple geographic coordinates to outline the study region.

In [5]:
#Select the temporal range
startDate = '2019-07-01'
endDate = '2019-09-30'

In [6]:
#Select the region of interest
geometry=ee.Geometry.Polygon(
        [[[-124.90409911061118, 48.86561659588255],
          [-124.37675536061118, 40.781579504457156],
          [-121.03691161061118, 34.95911886788346],
          [-116.81816161061118, 32.6220248379889],
          [-108.20488036061118, 31.129372813341096],
          [-103.28300536061118, 29.306756787018667],
          [-97.65800536061118, 25.563501734433476],
          [-93.79081786061118, 30.52559418727123],
          [-83.24394286061118, 30.0702803778401],
          [-81.13456786061118, 25.086840495546568],
          [-80.78300536061118, 31.579703334167082],
          [-74.45488036061118, 36.38701640672267],
          [-71.46659911061118, 41.44375403236037],
          [-66.36894286061118, 44.653999388182356],
          [-69.53300536061118, 47.814075148600786],
          [-83.24394286061118, 42.09923961801118],
          [-81.13456786061118, 44.653999388182356],
          [-88.51738036061118, 48.63381458991116],
          [-95.72441161061118, 48.86561659588255]]])

## Filtering and Visualizing RGB Sentinel-2 Data

In this section, we process Sentinel-2 satellite imagery using Google Earth Engine (GEE). The workflow includes:
1. **Filtering the Sentinel-2 Collection**:
   - Selects images within the specified time range.
   - Uses a predefined region of interest (ROI).
   - Applies a cloud filter to exclude images with more than 5% cloud cover.
   - Selects the **B4 (Red), B3 (Green), and B2 (Blue)** bands for true-color visualization.
   
2. **Visualization Setup**:
   - Defines parameters for displaying the imagery.
   - Uses the `geemap` library for interactive visualization.


In [7]:
#Filter the SENTINEL collection applying the date and geometry filter and cloud filter
s2_colection = ee.ImageCollection('COPERNICUS/S2_HARMONIZED').filterDate(startDate, endDate)\
        .filterBounds(geometry)\
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))\
        .select(['B4', 'B3', 'B2'])

In [8]:
#Create the visualization array
s2VisParams = {"bands": ['B4', 'B3', 'B2'], "min": 0, "max": 3000}

In [40]:
# Display the data
map = geemap.Map()
map.setCenter(-99.1332, 19.4326, 4);
map.add_layer(s2_colection, s2VisParams, 'Sentinel-2 Image');
map

Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

## **Filtering and Visualizing Sentinel-2 NIR, SWIR1, and SWIR2 Bands**

This section focuses on extracting **Near-Infrared (NIR), Shortwave Infrared 1 (SWIR1), and Shortwave Infrared 2 (SWIR2) bands** from Sentinel-2 imagery. These bands are crucial for applications such as **vegetation monitoring, water content analysis, and land cover classification**.

### **1. Counting the Available Sentinel-2 Images**
- The number of images in the filtered **Sentinel-2 collection** is retrieved using the `aggregate_array("system:index")` function.
- The total image count is printed to verify the dataset size.

### **2. Filtering the Sentinel-2 Collection for NIR, SWIR1, and SWIR2**
- A second collection (`NIR_collection`) is created from **Sentinel-2 Harmonized data**.
- The dataset is filtered by:
  - **Date range** (`startDate`, `endDate`)
  - **Region of interest** (`geometry`)
  - **Cloud cover** (only images with less than **5% cloud coverage**)
- The collection is further filtered to match the **indices (timestamps) of the original RGB dataset**, ensuring consistency in image availability.

### **3. Verifying the Filtered Collection**
- The number of images in the **NIR, SWIR1, and SWIR2** filtered collection is counted to ensure alignment with the RGB dataset.
- The count should be **identical to the total number of RGB images**.

### **4. Setting Up Visualization Parameters**
- A visualization configuration (`NIRVisParams`) is defined for displaying the **NIR (B8), SWIR1 (B11), and SWIR2 (B12)** bands.
- These bands highlight vegetation health, water bodies, and soil moisture.

### **5. Displaying the Image Collection**
- The `geemap` library is used to visualize the dataset interactively.
- The map is centered at predefined locations, and the processed **Sentinel-2 images** are added as a layer.

This workflow ensures a **consistent and well-filtered dataset** for further remote sensing analysis, allowing for accurate interpretation of land cover dynamics.


In [10]:
#1. Countering the available Sentinel-2 Images
image_ids = s2_colection.aggregate_array("system:index").getInfo()
print("Total images: " , len(image_ids))

Total images:  13026


In [11]:
#2. Filtering the Sentinel-2 Collection for NIR, SWIR1, and SWIR2
NIR_colection = ee.ImageCollection('COPERNICUS/S2_HARMONIZED').filterDate(startDate, endDate)\
        .filterBounds(geometry)\
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))

filter_index = ee.Filter.inList('system:index', s2_colection.aggregate_array('system:index'));
NIR = NIR_colection.filter(filter_index);

In [12]:
#3. Verifying the Filtered Collection
image_ids2 = NIR.aggregate_array("system:index").getInfo()
print("Total images: " , len(image_ids))

Total images:  13026


In [13]:
#4. Setting Up Visualization Parameters
NIRVisParams = {"bands": ["B8",'B11',"B12"], "min": 0, "max": 3000}

In [39]:
#5. Displaying the Image Collection
map = geemap.Map()
map.setCenter(-99.1332, 19.4326, 4);
map.add_layer(NIR, NIRVisParams, 'NIR_SWIR1_SWIR_2');
map

Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

## **Filtering and Visualizing Dynamic World Data**

In this section, we process **Dynamic World** data, a global land cover dataset developed by **Google and the World Resources Institute (WRI)**. This dataset provides near real-time, high-resolution land classification maps, offering valuable insights into land-use patterns.

### **1. Loading the Dynamic World Dataset**
- The dataset **GOOGLE/DYNAMICWORLD/V1** is loaded from Google Earth Engine (GEE).
- This dataset contains **nine land cover classes**, such as:
  - Water, Trees, Grass, Crops, Shrubland, Built-up areas, Bare ground, Snow/Ice, and Flooded vegetation.

### **2. Filtering the Dataset to Match Sentinel-2 Images**
- A filter is applied to **match the timestamps (system:index) of Sentinel-2 images**, ensuring temporal consistency between the datasets.
- This step ensures that **Dynamic World data aligns with Sentinel-2 images**, allowing for meaningful comparisons and analyses.

### **3. Setting Up Visualization Parameters**
- The visualization parameters (`dwVisParams`) define:
  - **Min-Max values** (`0-8`), corresponding to the land cover classes.
  - A **color palette** to visually distinguish different land cover types.

### **4. Displaying the Dynamic World Data**
- The **geemap** library is used to visualize the **Dynamic World classification** overlaid on a map.
- The map is centered at a specific location, ensuring the area of interest is displayed.
- The **land cover label** (`label`) is selected from the dataset and added as a visualization layer.

This workflow allows for **interactive exploration of land cover classifications**, providing essential insights for environmental and land-use analysis.


In [15]:
#1. Loading the Dynamic World Dataset
dw_colection = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')

In [16]:
#2. Filtering the Dataset to Match Sentinel-2 Images
index_filter = ee.Filter.inList('system:index', s2_colection.aggregate_array('system:index'));
dw = dw_colection.filter(index_filter);

In [17]:
#3. Setting Up Visualization Parameters
dwVisParams = {
  "min": 0,
  "max": 8,
  "palette": [
    '#419BDF', '#397D49', '#88B053', '#7A87C6', '#E49635', '#DFC35A',
    '#C4281B', '#A59B8F', '#B39FE1'
  ]
}

In [38]:
#4. Displaying the Dynamic World Data
map = geemap.Map()
map.setCenter(-99.1332, 19.4326, 4);
map.add_layer(dw.select("label"), dwVisParams, 'DW Image');
map

Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

## **Filtering Sentinel-2 Based on Available Dynamic World Images**

Since not all **Sentinel-2 images** have a corresponding **Dynamic World image**, we need to **filter Sentinel-2 again** to ensure both datasets are perfectly synchronized.

### **1. Counting the Available Dynamic World Images**
- The total number of **Dynamic World images** is retrieved and printed.
- This helps verify that we are working with a consistent dataset.

### **2. Filtering Sentinel-2 to Match Dynamic World**
- Sentinel-2 images are filtered based on the available **Dynamic World timestamps (`system:index`)**.
- This ensures that only **Sentinel-2 images with a matching Dynamic World classification** remain in the dataset.

### **3. Filtering the NIR, SWIR1, and SWIR2 Collection**
- The **NIR, SWIR1, and SWIR2** subset is also updated to match the filtered Sentinel-2 collection.
- This guarantees that all spectral bands are aligned across datasets.

### **4. Displaying the Filtered Sentinel-2 Data**
- The **geemap** library is used to visualize the filtered **Sentinel-2** dataset.
- The updated Sentinel-2 collection is displayed using the **s2VisParams** configuration.
- The visualization ensures that only images with **corresponding Dynamic World classifications** are included.

This step ensures that **Sentinel-2, Dynamic World, and their spectral subsets (NIR, SWIR1, SWIR2) are fully aligned**, avoiding inconsistencies in further analysis.


In [19]:
### 1. Counting the Available Dynamic World Images
image_ids = dw.aggregate_array("system:index").getInfo()
print("Total images: " , len(image_ids))
index_filter = ee.Filter.inList('system:index', dw.aggregate_array('system:index'));

Total images:  13024


In [20]:
### 2. Filtering Sentinel-2 to Match Dynamic World
s2 = s2_colection.filter(index_filter);
### 3. Filtering the NIR, SWIR1, and SWIR2 Collection
NIR=NIR.filter(index_filter);

In [37]:
### 4. Displaying the Filtered Sentinel-2 Data
map = geemap.Map()
map.setCenter(-99.1332, 19.4326, 4);
map.add_layer(s2, s2VisParams, 'Sentinel-2 Image');
map

Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

## **Creating the Sentinel-2 Composite**

In this section, we generate a **Sentinel-2 composite** by computing the **median** value of all available images. This process helps reduce noise, such as cloud coverage, and enhances the quality of the final dataset.

### **1. Computing the Median Composite**
- The median value is computed for:
  - **s2** → The full Sentinel-2 collection.
  - **NIR** → The filtered Near-Infrared (NIR, SWIR1, SWIR2) collection.
- This helps create a representative image with minimal noise and outliers.

### **2. Creating an RGB Visualization to Export**
- The `visualize` function is used to define the visualization settings for the median composite.
- The selected bands for visualization:
  - **True Color Composite (RGB)** → `B4 (Red)`, `B3 (Green)`, `B2 (Blue)`.
- This visualization is designed to be exported for further analysis and interpretation.

### **3. Creating a NIR, SWIR1, and SWIR2 Visualization to Export**
- A separate visualization is created for the **Near-Infrared (NIR) and Shortwave Infrared (SWIR1, SWIR2) bands**.
- The selected bands:
  - **NIR (B8)**, **SWIR1 (B11)**, and **SWIR2 (B12)** are useful for vegetation health assessment, water content analysis, and land cover classification.
- This visualization will be used for export, allowing further remote sensing applications.

### **4. Renaming the Bands for Clarity**
- The bands in the **NIR composite** are renamed to:
  - `"NIR"` → Corresponding to **B8**.
  - `"SWIR1"` → Corresponding to **B11**.
  - `"SWIR2"` → Corresponding to **B12**.
- This renaming makes the dataset more intuitive for further analysis.

By computing the **median composite**, we ensure that the final dataset represents a **clear, noise-free image** of the study area, improving its usability for land cover classification and environmental monitoring.


In [22]:
# 1. Computing the Median Composite
medianRGB = s2.median();
medianNir= NIR.median()

In [24]:
# 2. Visualizing the RGB Composite
visualization = medianRGB.visualize(**{
 "bands": ['B4', 'B3', 'B2'],
 "min": 0,
 "max": 3000
})

In [27]:
# 3. Visualizing the NIR, SWIR1, and SWIR2 Composite
NIRvisualization = medianNir.visualize(**{
 "bands": ['B8', 'B11', 'B12'],
 "min": 0,
 "max": 3000
})

In [28]:
# 4. Renaming the Bands for Clarity
NIRvisualization = NIRvisualization.rename(['NIR', 'SWIR1', 'SWIR2'])


## **Creating the Dynamic World Composite**

In this section, we generate a **Dynamic World (DW) composite** by computing the **mode** of all available classification images. This process ensures that the most frequently occurring land cover type is retained for each pixel, providing a reliable representation of the dominant land cover over time.

### **1. Selecting the Land Cover Labels**
- The **"label"** band is selected from the **Dynamic World dataset**, representing the land cover classification.

### **2. Computing the Mode Composite**
- The **mode** is calculated using `ee.Reducer.mode()`, which determines the most common land cover class for each pixel across all images.
- This method helps reduce noise and ensures that the final composite represents a stable classification.

By computing the **mode composite**, we create a **stable, noise-free land cover classification**, making it useful for environmental monitoring and change detection studies.


In [29]:
# 1. Selecting the Land Cover Labels
classification = dw.select('label');

In [30]:
# 2. Computing the Mode Composite
dwComposite = classification.reduce(ee.Reducer.mode());

### **3. Displaying the Composites**
The `geemap` library is used to visualize the **Sentinel-2** and **Dynamic World** composites in an interactive map.

- **Sentinel-2 RGB Composite** (`median`): Displays a true-color image of the region.
- **Sentinel-2 NIR-SWIR Composite** (`medianNir`): Enhances features related to vegetation, water bodies, and land cover changes.
- **Dynamic World Composite** (`dwComposite`): Shows the final **land cover classification**, ensuring consistency over time.

The map is then centered on the region of interest to facilitate exploration and analysis.

In [36]:
# 3. Displaying the Composites
map = geemap.Map()
map.setCenter(-99.1332, 19.4326, 4);
map.add_layer(medianRGB, s2VisParams, 'Sentinel-2 Image');
map.add_layer(medianNir, NIRVisParams, 'Nir-Sentinel-2 image');
map.add_layer(dwComposite, dwVisParams, 'DW composite');
map

Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

## **Dividing the Region into Smaller Areas for Export**

Since the **Region of Interest (ROI)** is too large to export as a single image, we divide it into **smaller regions**. This process allows for efficient data handling and ensures that the images can be exported without exceeding size limitations.

---

### **1. Visualizing the Original ROI**
- The original **polygon (geometry)** defining the ROI is displayed on the map.
- This step provides a visual reference for the selected study area before subdivision.

---

### **2. Extracting ROI Coordinates**
- The **coordinates** of the ROI are extracted using `geometry.getInfo()["coordinates"][0]`.
- The number of coordinates is counted to ensure accuracy.
- Each coordinate is converted into an **Earth Engine Feature**, representing **individual points**.

---

### **3. Creating a Feature Collection of Points**
- The extracted points are stored as a **FeatureCollection**, which allows visualization and further spatial operations.
- The points are displayed on the map to confirm their alignment with the original ROI.

---

### **4. Defining the Grid Size for Subregions**
- A **grid-based subdivision** is created by defining:
  - `Lx`: The step size in the longitude direction.
  - `Ly`: The step size in the latitude direction.
- These values define the **size of the smaller regions** to be exported.

---

### **5. Generating the First Subregion**
- A **small bounding box** is created using the first coordinate of the ROI.
- Four corner points are generated to define a **rectangular subregion**.
- The resulting subregion is displayed on the map for verification.

---

### **6. Creating a Grid to Cover the Entire ROI**
- The **bounding box** of the ROI is computed using `geometry.bounds()`.
- The **minimum and maximum latitude and longitude** values are extracted.
- A matrix of points is generated to define **grid-based subdivisions**.

---

### **7. Constructing the Grid of Subregions**
- A list of **grid coordinates** is generated using latitude and longitude steps.
- Each coordinate pair defines a **rectangular subregion**.
- Only the rectangles that **fall within the ROI** are kept.
- The final list of rectangles is converted into a **FeatureCollection** for visualization.

---

### **8. Displaying the Final Grid on the Map**
- The **original ROI**, **extracted points**, and **generated subregions** are visualized on an interactive map.
- The final grid ensures that the entire **ROI is covered with non-overlapping smaller regions**, optimizing the export process.

This process provides an efficient way to **divide large regions into smaller, manageable sections**, making it easier to export and analyze high-resolution satellite imagery. 🚀


In [35]:
# 1. Visualizing the Original ROI
map = geemap.Map()
map.setCenter(-99.1332, 19.4326, 4);
map.addLayer(geometry, {'color': 'FF0000'}, 'Polígono')
map

Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

In [33]:
# 2. Extracting ROI Coordinates
coord_points=[]
geometry_coord=geometry.getInfo()["coordinates"][0]
for i,cord in enumerate(geometry_coord):
  coord_points.append(ee.Feature(ee.Geometry.Point(cord),{"name":f"punto{i}"}))
print(len(coord_points))

20


In [34]:
# 3. Creating a Feature Collection of Points
points=ee.FeatureCollection(coord_points)
map = geemap.Map()
map.setCenter(-99.1332, 19.4326, 4);
map.addLayer(geometry, {'color': '000000'}, 'Polígono')
map.addLayer(points, {'color': '1A237E'}, 'Puntos')
map

Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

In [42]:
# 4. Defining the Grid Size for Subregions
Lx=1.93359375
Ly=1.0811553610059477

In [44]:
# 5. Generating the First Subregion
list_region_coordinates=[]
coord1=geometry_coord[0]

coord2=[coord1[0],coord1[1]-Ly]
coord3=[coord2[0]+Lx,coord2[1]]
coord4=[coord1[0]+Lx,coord1[1]]

list_region_coordinates.append(coord1)
list_region_coordinates.append(coord2)
list_region_coordinates.append(coord3)
list_region_coordinates.append(coord4)

region_of_interest=ee.Geometry.Polygon(list_region_coordinates)

map = geemap.Map()
map.setCenter(-99.1332, 19.4326, 4);
map.addLayer(geometry, {'color': 'FF0000'}, 'Polígono')
map.addLayer(points, {'color': '000000'}, 'Puntos')
map.addLayer(region_of_interest, {'color': '1A237E'}, 'ROI')
map

Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

In [50]:
# 6. Creating a Grid to Cover the Entire ROI
exterior = geometry.bounds()

longitud_step = 1.0
latitud_step = 1.0

ext_coords = exterior.coordinates().get(0)

lat_min = ext_coords.getInfo()[1][1]
lat_max = ext_coords.getInfo()[3][1]
lon_min = ext_coords.getInfo()[0][0]
lon_max = ext_coords.getInfo()[2][0]

In [51]:
# 7. Constructing the Grid of Subregions
longitud_step = 1.0
latitud_step = 1.0

coords = []
for lat in range(int(lat_min), int(lat_max), int(latitud_step)):
    for lon in range(int(lon_min), int(lon_max), int(longitud_step)):
        coords.append([lon, lat])

rectangles = []
for coord in coords:
    rect = ee.Geometry.Rectangle(
        [coord[0], coord[1], coord[0] + longitud_step, coord[1] + latitud_step])
    if geometry.contains(rect).getInfo():
        rectangles.append(rect)

regions = ee.FeatureCollection(rectangles)

print('Número de regiones:', len(rectangles))

Número de regiones: 797


In [53]:
# 8. Displaying the Final Grid on the Map
map = geemap.Map()
map.setCenter(-99.1332, 19.4326, 4);
map.addLayer(geometry, {'color': 'FF0000'}, 'Polígono')
map.addLayer(points, {'color': '000000'}, 'Puntos')
map.addLayer(regions, {'color': '1A237E'}, 'ROI')
map

Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

## **Exporting the Composites and Regions to Google Drive**

In this section, we export the generated composites from **Sentinel-2** and **Dynamic World**, as well as the **Feature Collection of subregions**, to **Google Drive** for further analysis.

### **1. Exporting the Dynamic World Composite**
- The **Dynamic World composite** (`dwComposite`) is exported as a set of image tiles.
- Each tile corresponds to one of the **subregions** (`rectangles`).
- The export is done using **`ee.batch.Export.image.toDrive`**, specifying:
  - `"scale": 10` → Defines the resolution of the exported image (10m per pixel).
  - `"maxPixels": 1e10` → Ensures large images can be exported without hitting pixel limits.
  - `"folder": "DW_COMPOSITES_2016"` → The folder where the images will be stored in Google Drive.

### **2. Exporting the Sentinel-2 RGB Composite**
- The **Sentinel-2 true-color composite** (`visualization`) is exported.
- Each tile corresponds to one of the **subregions**.
- The images are saved in the folder `"S2_COMPOSITES_2016"` with descriptive filenames.

### **3. Exporting the Sentinel-2 NIR, SWIR1, and SWIR2 Composite**
- The **Near-Infrared (NIR), Shortwave Infrared 1 (SWIR1), and Shortwave Infrared 2 (SWIR2)** composite (`NIRvisualization`) is exported.
- The folder `"S2_MULTI_COMPOSITES_2024"` is used to store these images.

By exporting the composites in **smaller tiles**, we ensure that the data is more manageable and can be used effectively for remote sensing applications. 🚀


In [ ]:
for i,region in enumerate(rectangles):
    # 1. Exporting the Dynamic World Composite
    Dw_task=ee.batch.Export.image.toDrive(**{
          "image": dwComposite,
          "description" : f"Dynamic world COMPOSITE 2016-07-01 to 2016-09-30 Part{i}",
          "folder":"DW_COMPOSITES_2016",
          "scale":10,
          "region":region,
          "maxPixels": 1e10
        })
    Dw_task.start()

    # 2. Exporting the Sentinel-2 RGB Composite
    S2_task=ee.batch.Export.image.toDrive(**{
          "image": visualization,
          "description" : f"SENTINEL COMPOSITE 2016-07-01 to 2016-09-30 Part{i}",
          "folder":"S2_COMPOSITES_2016",
          "scale":10,
          "region":region,
          "maxPixels": 1e10
        })
    S2_task.start()

    # 3. Exporting the Sentinel-2 NIR, SWIR1, and SWIR2 Composite
    SWIT_NIR_task=ee.batch.Export.image.toDrive(**{
          "image": NIRvisualization,
          "description" : f"SENTINEL NIR,SWIR COMPOSITE 2024-07-01 to 2024-09-30 Part{i}",
          "folder":"S2_MULTI_COMPOSITES_2024",
          "scale":10,
          "region":region,
          "maxPixels": 1e10
        })
    SWIT_NIR_task.start()
